In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:

generators_as_run = ["BT_name", "PL_name", "PL_type", "BT_type", "CS B=C", "CS B=W"]
generator_relabeling = {"BT_name": "Name-BT", "PL_name":"Name-PL", "PL_type":"Slate-PL", 
                        "BT_type":"Slate-BT", "CS B=C":"Slate-CS (C)", "CS B=W": "Slate-CS (W)"}

generator_reverse_dict = {v:k for k,v in generator_relabeling.items()}

generators = list(generator_relabeling.values())
pi_values = [.5, .6, .7, .8, .9, .99]
pi_types_as_run= [ "strong", "uniform", "mixed", "Y"]
pi_types_relabeling = {"uniform": "U", "strong" : "S", "mixed": "X", "Y":"Y"}
pi_types = list(pi_types_relabeling.values())


gen_colors= {"Slate-CS (W)":'black',
         "Slate-CS (C)":'#FB607F',
         "Slate-BT":'#8DB600',
         "Slate-PL":'#FFBF00',
         "Name-BT": '#E32636',
         "Name-PL": '#1560BD'}

colors= ['#E32636', '#1560BD', '#FFBF00', '#8DB600', '#FB607F', "black"]

In [3]:
def compute_stats_full_ballots(g):
    bloc_to_count = "B"

    headed_by = {"A": {"borda": 0, "second": 0, "third": 0},
                         "B": {"borda": 0, "second": 0, "third": 0}}
    

    with open(f'ranked_marginals_profiles/{g}_{pi_type}_n_{n}_pi_{pi}.pkl', 'rb') as file:
        ballot_dict = pickle.load(file)

    slate_to_candidates = {"A": [f"A{i}" for i in range(n)],
            "B": [f"B{i}" for i in range(n)]}

    candidates_to_slates = {c:b for b,c_list in slate_to_candidates.items() for c in c_list}

    # convert from candidate names to slate names
    ballot_frequencies = {}
    for b,f in ballot_dict.items():
        ballot_by_slate  = tuple([candidates_to_slates[c] for c in b])

        if ballot_by_slate in ballot_frequencies.keys():
            ballot_frequencies[ballot_by_slate] += f
        
        else:
            ballot_frequencies[ballot_by_slate] = f
    
    num_ballots = float(sum(ballot_frequencies.values()))

    first_place = sum([f for b,f in ballot_frequencies.items() if b[0] == bloc_to_count])/num_ballots
    second_place = sum([f for b,f in ballot_frequencies.items() if b[1] == bloc_to_count])/num_ballots

    third_total = float(sum([f for b,f in ballot_frequencies.items() if b[:2].count(bloc_to_count) < n]))
    third = sum([f for b,f in ballot_frequencies.items() if b[2] == bloc_to_count])
    
    if third_total != 0:
        third_place = third/third_total
    else:
        third_place = np.nan

    ballot_length = len(list(ballot_dict.keys())[0])
    total_borda = float(sum(
                    [sum(
                        [1*(ballot_length-i) for i, cand in enumerate(b[:ballot_length])])
                        *f 
                        for b,f in ballot_frequencies.items()]
                        ))
    borda = sum(
                [sum(
                    [1*(ballot_length-i) for i, cand in enumerate(b[:ballot_length]) if cand == bloc_to_count])
                    *f 
                    for b,f in ballot_frequencies.items()]
                    )/total_borda
    

    for header in ["A", "B"]:
        total_ballots = float(sum([f for b,f in ballot_frequencies.items() if b[0] == header]))

        if total_ballots != 0:
            total_borda = sum(
                        [sum(
                        [1*(ballot_length-i) for i, cand in enumerate(b[:ballot_length])])
                        *f 
                        for b,f in ballot_frequencies.items() if b[0] == header]
                        )
            borda_w = sum(
                        [sum(
                        [1*(ballot_length-i) for i, cand in enumerate(b[:ballot_length]) if cand == bloc_to_count])
                        *f 
                        for b,f in ballot_frequencies.items() if b[0] == header]
                        )
            
            headed_by[header]["borda"] = borda_w/float(total_borda)
            
            
            second_w = sum([f for b,f in ballot_frequencies.items() if b[0] == header and b[1] == bloc_to_count ])
            headed_by[header]["second"] = second_w/total_ballots
            
            # need to make sure we haven't exhausted the W candidates yet
            third_w = sum([f for b,f in ballot_frequencies.items() if b[0] == header and b[2] == bloc_to_count])
            
            third_total = sum([f for b,f in ballot_frequencies.items() if b[0] == header and b[:2].count(bloc_to_count)<n])

            if third_total == 0:
                headed_by[header]["third"] = np.nan
            else:
                headed_by[header]["third"] = float(third_w/third_total)
            
            
        else:
            headed_by[header]["borda"] = np.nan
            headed_by[header]["second"] = np.nan
            headed_by[header]["third"] = np.nan
    return first_place, second_place, third_place,borda, headed_by

def compute_stats_incomplete_ballots(g, n):
    """
    n is number of candidates per bloc
    """
    bloc_to_count ="B"
    headed_by = {"A": {"borda": 0, "second": 0, "third": 0},
                         "B": {"borda": 0, "second": 0, "third": 0}}

    
    with open(f'ranked_marginals_profiles/{g}_{pi_type}_n_{n}_pi_{pi}.pkl', 'rb') as file:
        ballot_dict = pickle.load(file)

    slate_to_candidates = {"A": [f"A{i}" for i in range(n)],
            "B": [f"B{i}" for i in range(n)]}

    candidates_to_slates = {c:b for b,c_list in slate_to_candidates.items() for c in c_list}

    # convert from candidate names to slate names
    ballot_frequencies = {}
    for b,f in ballot_dict.items():
        ballot_by_slate  = tuple([candidates_to_slates[c] for c in b])

        if ballot_by_slate in ballot_frequencies.keys():
            ballot_frequencies[ballot_by_slate] += f
        
        else:
            ballot_frequencies[ballot_by_slate] = f
    
    num_ballots = float(sum(ballot_frequencies.values()))

    first_place = sum([f for b,f in ballot_frequencies.items() if b[0] == bloc_to_count])/num_ballots

    num_non_bullet= float(sum([f for b,f in ballot_frequencies.items() if len(b) > 1]))
    second_place = sum([f for b,f in ballot_frequencies.items() if len(b)>1 and b[1] == bloc_to_count])/num_non_bullet

    third_total = float(sum([f for b,f in ballot_frequencies.items() if len(b)>2 and b[:2].count(bloc_to_count) < n]))
    third = sum([f for b,f in ballot_frequencies.items() if len(b)>2 and b[2] == bloc_to_count])
    
    if third_total != 0:
        third_place = third/third_total
    else:
        third_place = np.nan

    max_ballot_length = 2 * n
    total_borda = float(sum(
                    [sum(
                        [1*(max_ballot_length-i) for i, cand in enumerate(b)])
                        *f 
                        for b,f in ballot_frequencies.items()]
                        ))
    borda = sum(
                [sum(
                    [1*(max_ballot_length-i) for i, cand in enumerate(b) if cand == bloc_to_count])
                    *f 
                    for b,f in ballot_frequencies.items()]
                    )/total_borda
    

    for header in ["A", "B"]:
        total_ballots = float(sum([f for b,f in ballot_frequencies.items() if b[0] == header]))

        if total_ballots != 0:
            total_borda = sum(
                        [sum(
                        [1*(max_ballot_length-i) for i, cand in enumerate(b)])
                        *f 
                        for b,f in ballot_frequencies.items() if b[0] == header]
                        )
            borda_w = sum(
                        [sum(
                        [1*(max_ballot_length-i) for i, cand in enumerate(b) if cand == bloc_to_count])
                        *f 
                        for b,f in ballot_frequencies.items() if b[0] == header]
                        )
            
            headed_by[header]["borda"] = borda_w/float(total_borda)
            
            num_non_bullet = float(sum([f for b,f in ballot_frequencies.items() if b[0] == header and len(b)>1]))
            second_w = sum([f for b,f in ballot_frequencies.items() if len(b)>1 and b[0] == header and b[1] == bloc_to_count ])
            headed_by[header]["second"] = second_w/num_non_bullet
            
            # need to make sure we haven't exhausted the A candidates yet
            third_w = sum([f for b,f in ballot_frequencies.items() if len(b) > 2 and b[0] == header and b[2] == bloc_to_count])
            
            third_total = sum([f for b,f in ballot_frequencies.items() if len(b) > 2 and b[0] == header and b[:2].count(bloc_to_count)<n])

            if third_total == 0:
                headed_by[header]["third"] = np.nan
            else:
                headed_by[header]["third"] = float(third_w/third_total)
            
            
        else:
            headed_by[header]["borda"] = np.nan
            headed_by[header]["second"] = np.nan
            headed_by[header]["third"] = np.nan

    return first_place, second_place, third_place,borda, headed_by
    

In [4]:
dfs = {}
for n in range(2,6):
    for pi_type in pi_types_as_run:
        for pi in [.5, .6, .7, .8, .9, .99]:
            # all in terms of A candidates
            first_place = []
            second_place = []
            third_place = []
            borda = []

            # given ballot starts with W/C, tally for W
            headed_by = {"A": {"borda": [], "second": [], "third": []},
                         "B": {"borda": [], "second": [], "third": []}}
            

            for g in generators_as_run:
                if "CS" in g:
                    f, s, t,b, hb = compute_stats_incomplete_ballots(g,n)
                else:
                    f, s, t,b, hb = compute_stats_full_ballots(g)
                first_place.append(f)
                second_place.append(s)
                third_place.append(t)
                borda.append(b)

                for header in ["A", "B"]:
                    for stat in  ["borda", "second", "third"]:
                        headed_by[header][stat].append(hb[header][stat])
                


            df = pd.DataFrame({"First place votes for B":first_place,
                               "Second place votes for B":second_place ,
                               "Third place votes for B": third_place,
                               "Borda score for B": borda,
                               "Borda score for B given A first":headed_by["A"]["borda"],
                               "Second place votes for B given A first":headed_by["A"]["second"],
                               "Third place votes for B given A first":headed_by["A"]["third"],
                               "Borda score for B given B first":headed_by["B"]["borda"],
                               "Second place votes for B given B first":headed_by["B"]["second"],
                               "Third place votes for B given B first":headed_by["B"]["third"],
                               }, 
                               index = generators_as_run)
            
            dfs[f"n_{n}_{pi_types_relabeling[pi_type]}_pi_{pi}"] = df

In [5]:
for stat in list(dfs.values())[0].columns:
    fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(16, 12))

    for i, pi_type in enumerate(pi_types):
        for j, n in enumerate(range(2, 6)):
            for g in generators:
                # accesses first place votes for each generator and fixed PI_type and num_cands
                data = pd.Series([dfs[f"n_{n}_{pi_type}_pi_{pi}"].loc[generator_reverse_dict[g], stat] for pi in pi_values], index = pi_values)
                axes[i, j].plot(data, label = g, color = gen_colors[g], lw =2)
                                
                axes[i,j].set_title(f"{n} candidates, {pi_type} preference")
                axes[i,j].set_xlabel("pi")
                axes[i,j].set_ylabel(stat)
                axes[i,j].set_ylim(0,1.1)

    fig.suptitle(f"B bloc profile\n{stat} as a function of pi")
    plt.legend(bbox_to_anchor=(1,1))
    plt.tight_layout()
    # plt.show()
    plt.savefig(f"figures/{stat}_by_cands_and_pref")
    plt.close()

In [6]:
n_colors = {n: colors[i] for i,n in enumerate(range(2,6))}

for stat in list(dfs.values())[0].columns:
    fig, axes = plt.subplots(nrows=4, ncols=6, figsize=(20, 11))

    for i, pi_type in enumerate(pi_types):
        for j, g in enumerate(generators):
            for _, n in enumerate(range(2, 6)):
                # accesses first place votes for each generator and fixed PI_type and num_cands
                data = pd.Series([dfs[f"n_{n}_{pi_type}_pi_{pi}"].loc[generator_reverse_dict[g], stat] for pi in pi_values], index = pi_values)
                axes[i, j].plot(data, label = n, color = n_colors[n], lw=2)
                
                axes[i,j].set_title(f"{g}, {pi_type} preference")
                axes[i,j].set_xlabel("pi")
                axes[i,j].set_ylabel(stat)
                axes[i,j].set_ylim(0,1.1)
            

    fig.suptitle(f"B bloc profile\n{stat} as a function of pi")
    plt.legend(bbox_to_anchor=(1,1))
    plt.tight_layout()
    # plt.show()
    plt.savefig(f"figures/{stat}_by_generator_and_pref")
    plt.close()

In [7]:
pi_colors = {pi_type: colors[i] for i, pi_type in enumerate(pi_types)}

for stat in list(dfs.values())[0].columns:
    fig, axes = plt.subplots(nrows=6, ncols=4, figsize=(16, 20))

    # for i, pi_type in enumerate(["strong", "uniform"]):
    for i, g in enumerate(generators):
        for j, n in enumerate(range(2,6)):
            for pi_type in pi_types:
                data = pd.Series([dfs[f"n_{n}_{pi_type}_pi_{pi}"].loc[generator_reverse_dict[g], stat] for pi in pi_values], index = pi_values)
                axes[i, j].plot(data, label = pi_type, color = pi_colors[pi_type])
                
                axes[i,j].set_title(f"{g}, {n} candidates")
                axes[i,j].set_xlabel("pi")
                axes[i,j].set_ylabel(stat)
                axes[i,j].set_ylim(0,1.1)
            # axes[i,j].legend()

    fig.suptitle(f"B bloc profile\n{stat} as a function of pi\n")
    plt.legend(bbox_to_anchor = (1,1))
    plt.tight_layout()
    # plt.show()
    plt.savefig(f"figures/{stat}_by_cands_and_generator")
    plt.close()